# ENSTA — Séance 1 · Observer, représenter, apprendre

**Introduction à l’apprentissage profond · dernière année · septembre 2026**  
**Version étudiante · exercices à compléter · Python / PyTorch · processeur CPU suffisant**

Ce notebook accompagne la première séance consacrée à l’apprentissage à partir de données. Son fil directeur est volontairement méthodique : **avant de construire un modèle, il faut regarder les observations ; avant d’interpréter un score, il faut comprendre le protocole qui l’a produit ; avant de faire confiance à un gradient, il faut savoir quelle quantité il dérive.**

Nous commencerons par le **quartet d’Anscombe**. Quatre jeux de données possèdent des résumés numériques presque identiques, mais leurs représentations graphiques racontent des histoires radicalement différentes. Cette entrée en matière sert à reprendre les vecteurs, les matrices, les axes de calcul et les statistiques descriptives, tout en installant une règle qui restera valable pendant toute la séance :

> **prévoir → exécuter → observer → expliquer.**

La suite conserve les cinq séquences pratiques du cours. Un problème de classification de type XOR permet de distinguer **représentation**, **optimisation** et **généralisation** ; un oscillateur amorti montre enfin comment une équation différentielle peut devenir une source d’information pour un réseau. Ce dernier exemple est un cas d’étude contrôlé : il ne cherche pas à concurrencer un solveur classique, mais à rendre visibles la construction et les limites d’un PINN.

| Séquence pratique | Repère dans la séance de 5 h | Temps | Question centrale |
|---|---:|---:|---|
| [Prélude — quartet d’Anscombe](#anscombe) | 00:00–00:20 | 20 min | Que perd-on lorsqu’on résume sans représenter ? |
| [A — Tenseurs et protocole](#tp-a) | 00:40–01:00 | 20 min | Comment préparer les données sans fuite ? |
| [B — Linéaire ou non linéaire ?](#tp-b) | 01:25–01:55 | 30 min | Quand faut-il apprendre une représentation ? |
| [C — Ouvrir la boîte du gradient](#tp-c) | 02:35–03:05 | 30 min | Que calcule exactement la rétropropagation ? |
| [D — Choisir sans regarder le test](#tp-d) | 03:30–03:55 | 25 min | Comment protéger une conclusion expérimentale ? |
| [E — L’oscillateur comme contrainte](#tp-e) | 04:25–04:55 | 30 min | Comment une loi physique devient-elle une perte ? |
| [Ticket de sortie](#sortie) | 04:55–05:00 | 5 min | Relier données, modèle, gradient et validation |

Les plages intermédiaires sont consacrées au cours et aux deux pauses. Travaillez en binôme : une personne tient le clavier, l’autre annonce à voix haute la forme attendue de chaque tenseur, formule une prédiction et vérifie ensuite que le résultat répond bien à la question posée.

### Comment lire un exercice

Chaque exercice est écrit selon un même gabarit, de façon que vous sachiez toujours où trouver l’information dont vous avez besoin.

**Objectif** annonce en une phrase ce que l’exercice doit vous faire comprendre ; **Ce que vous devez écrire** décrit précisément la fonction attendue, ses arguments, la forme de ce qu’elle reçoit et la forme de ce qu’elle renvoie ; **Marche à suivre** découpe le travail en étapes courtes, dont chacune correspond à une ou deux lignes de code ; **Vérification** indique ce que contrôle la cellule de test qui suit, et donc ce que signifie l’absence d’erreur ; **Erreurs fréquentes** rassemble les confusions que ce type d’exercice provoque habituellement, afin que vous puissiez les reconnaître au lieu de les subir ; **À expliquer** pose enfin la question d’interprétation que vous devez savoir traiter oralement, parce que c’est elle, et non la ligne de code, qui sera réutilisée dans la suite du module.

Aucune étape ne suppose une astuce : si une consigne vous paraît obscure, relisez la forme attendue, puis écrivez-la en commentaire avant de coder. La quasi-totalité des erreurs de cette séance sont des erreurs de dimensions, et une dimension écrite à la main se corrige beaucoup plus vite qu’une dimension devinée.

### Règles de travail

Les cellules marquées `TODO` s’interrompent volontairement sur `NotImplementedError` tant que la réponse n’a pas été écrite : ce comportement ne signale pas une panne, mais une question qui vous attend. Les assertions placées après les exercices sont des contrôles locaux ; elles vérifient une propriété précise et ne remplacent ni l’examen des figures ni l’interprétation des résultats. Exécutez les cellules dans l’ordre, car chaque partie réutilise les objets construits par la précédente.


## Démarrage — à effectuer avant la séance si possible

Ouvrez le notebook dans JupyterLab, VS Code ou Google Colab. Dans Colab, importez le fichier ou ouvrez-en une copie dans votre Drive, choisissez un environnement **Python 3 / CPU**, exécutez les cellules dans l’ordre et sauvegardez régulièrement votre copie, l’état d’exécution du runtime étant temporaire. Aucun GPU n’est nécessaire : toutes les expériences de la séance tiennent sur un processeur ordinaire, et aucune donnée n’est téléchargée.

La commande d’installation de la cellule suivante est désactivée ; ne la décommentez que si un import échoue, et redémarrez alors le noyau. La visualisation Plotly du quartet d’Anscombe est facultative : si Plotly n’est pas disponible, la figure Matplotlib demeure suffisante pour réaliser l’activité.

Une dernière recommandation, qui vaut pour les cinq heures : lorsque la cellule suivante affiche un message d’erreur, lisez-en d’abord la **dernière** ligne, qui nomme l’erreur, puis remontez jusqu’à la ligne de votre code qui l’a déclenchée. Un `RuntimeError` mentionnant deux formes incompatibles se résout presque toujours en écrivant les dimensions des deux opérandes.


In [ ]:
# À décommenter seulement si nécessaire, puis redémarrer le noyau.
# %pip install torch numpy matplotlib scipy


In [ ]:
import copy
import math
import platform
import time
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import TensorDataset, DataLoader

SEED = 2026
DEVICE = torch.device("cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(1)  # petits tenseurs : limiter le surcoût du parallélisme
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
NOTEBOOK_START = time.perf_counter()

def export_figure(fig, name):
    # Option pour l'enseignant : définir ENSTA_FIGURE_DIR pour exporter les figures.
    destination = os.environ.get("ENSTA_FIGURE_DIR")
    if destination:
        directory = Path(destination)
        directory.mkdir(parents=True, exist_ok=True)
        fig.savefig(directory / f"{name}.pdf", bbox_inches="tight")
        fig.savefig(directory / f"{name}.png", dpi=160, bbox_inches="tight")
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        for index, axis in enumerate(fig.axes, start=1):
            bbox = axis.get_tightbbox(renderer).transformed(fig.dpi_scale_trans.inverted()).expanded(1.05, 1.10)
            # Masquer les axes voisins pour que leurs textes ne débordent pas dans le recadrage.
            visibility = [other.get_visible() for other in fig.axes]
            for other in fig.axes:
                other.set_visible(other is axis)
            fig.savefig(directory / f"{name}_panel{index}.pdf", bbox_inches=bbox)
            fig.savefig(directory / f"{name}_panel{index}.png", dpi=160, bbox_inches=bbox)
            for other, was_visible in zip(fig.axes, visibility):
                other.set_visible(was_visible)
print(f"Python {platform.python_version()} | PyTorch {torch.__version__} | {DEVICE}")
print("Les graines fixées facilitent la comparaison ; une identité bit à bit entre machines n'est pas garantie.")


<a id="anscombe"></a>
## Prélude — Le quartet d’Anscombe : voir les données avant de les modéliser · 20 min

En 1973, le statisticien Francis Anscombe a construit quatre petits jeux de données pour établir une idée simple et durable : **des résumés numériques presque identiques peuvent correspondre à des structures très différentes**. Le but n’est pas de dévaloriser la moyenne, la variance, la corrélation ou la régression linéaire, qui répondent chacune à une question précise. Le danger apparaît lorsqu’on leur demande de remplacer l’examen de la structure des observations.

Ce prélude poursuit deux objectifs simultanés. Sur le fond, il installe la règle de méthode qui gouvernera la séance entière. Sur la forme, il sert de reprise en main des tenseurs : indexation, choix d’un axe d’agrégation, diffusion des dimensions et vectorisation d’un calcul statistique.

**Convention de représentation.** Chaque jeu contient onze couples $(x_i,y_i)$. Un jeu sera donc une matrice de forme `(11, 2)` et les quatre jeux formeront un tenseur de forme `(4, 11, 2)`, dans lequel l’axe 0 désigne le jeu, l’axe 1 l’observation et l’axe 2 la variable, $x$ pour l’indice 0 et $y$ pour l’indice 1. Retenez cette lecture : une dimension n’est pas un simple nombre, elle porte une **signification expérimentale**, et c’est cette signification qui dicte l’axe sur lequel on agrège.

La cellule suivante construit le tenseur et affiche trois formes. Avant de l’exécuter, annoncez à votre binôme ce que vous attendez pour chacune d’elles.


In [ ]:
ANSCOMBE = torch.tensor([
    [[10., 8.04], [8., 6.95], [13., 7.58], [9., 8.81], [11., 8.33],
     [14., 9.96], [6., 7.24], [4., 4.26], [12., 10.84], [7., 4.82], [5., 5.68]],
    [[10., 9.14], [8., 8.14], [13., 8.74], [9., 8.77], [11., 9.26],
     [14., 8.10], [6., 6.13], [4., 3.10], [12., 9.13], [7., 7.26], [5., 4.74]],
    [[10., 7.46], [8., 6.77], [13., 12.74], [9., 7.11], [11., 7.81],
     [14., 8.84], [6., 6.08], [4., 5.39], [12., 8.15], [7., 6.42], [5., 5.73]],
    [[8., 6.58], [8., 5.76], [8., 7.71], [8., 8.84], [8., 8.47],
     [8., 7.04], [8., 5.25], [19., 12.50], [8., 5.56], [8., 7.91], [8., 6.89]],
], dtype=torch.float64)
ANSCOMBE_LABELS = ("I", "II", "III", "IV")

print("Forme du tenseur complet :", tuple(ANSCOMBE.shape))
print("Un jeu de données :", tuple(ANSCOMBE[0].shape))
print("Une observation :", tuple(ANSCOMBE[0, 0].shape), "->", ANSCOMBE[0, 0].tolist())


### 0.1 — Indexer un tenseur en donnant un sens à chaque axe · 4 min

**Objectif.** Savoir traduire une phrase française décrivant une donnée en une expression d’indexation, et prévoir la forme du résultat avant de l’obtenir.

**Ce que vous devez écrire.** Trois variables extraites du tenseur `ANSCOMBE`, de forme `(4, 11, 2)`, en utilisant uniquement l’indexation, sans boucle et sans recopie manuelle de valeurs :

`jeu_I` doit contenir la matrice complète du premier jeu, de forme `(11, 2)` ; `tous_les_x` doit contenir les abscisses des quatre jeux, de forme `(4, 11)` ; `y_jeu_III` doit contenir le vecteur des ordonnées du troisième jeu, de forme `(11,)`.

**Marche à suivre.** Rappelez-vous que l’expression `T[a, b, c]` sélectionne le long des trois axes dans l’ordre, et que le symbole `:` conserve un axe en entier. Fixer un axe avec un entier **supprime** cet axe du résultat, tandis que l’écrire `:` le conserve. Pour chaque variable, commencez donc par écrire en commentaire la phrase « je fixe l’axe … à … et je conserve les axes … », puis traduisez-la. Attention au décalage habituel entre la numérotation des jeux, de I à IV, et les indices de Python, de 0 à 3 : le troisième jeu porte l’indice 2.

**Vérification.** Les trois assertions contrôlent exactement les formes annoncées ci-dessus ; si elles passent, vos axes sont correctement ordonnés. Les valeurs affichées ensuite vous permettent de vérifier d’un coup d’œil que `y_jeu_III` contient bien des ordonnées, comprises entre 5 et 13, et non des abscisses.

**Erreurs fréquentes.** Confondre `ANSCOMBE[0, :, 1]`, qui donne les ordonnées du premier jeu, avec `ANSCOMBE[:, 0, 1]`, qui donne l’ordonnée de la première observation des quatre jeux ; oublier que l’indexation par un entier supprime un axe, et obtenir `(4, 11, 1)` au lieu de `(4, 11)` ; indexer le quatrième jeu en écrivant l’indice 4.


In [ ]:
# TODO 0.1 — remplacer les trois valeurs None par des indexations de ANSCOMBE.
# Rappel des axes : (jeu, observation, variable) avec variable = 0 pour x, 1 pour y.

jeu_I = None        # attendu : toutes les observations, les deux variables, du jeu d'indice 0 -> (11, 2)
tous_les_x = None   # attendu : tous les jeux, toutes les observations, la variable 0    -> (4, 11)
y_jeu_III = None    # attendu : le jeu d'indice 2, toutes les observations, la variable 1 -> (11,)

assert jeu_I.shape == (11, 2)
assert tous_les_x.shape == (4, 11)
assert y_jeu_III.shape == (11,)
print("jeu_I :", tuple(jeu_I.shape))
print("tous_les_x :", tuple(tous_les_x.shape))
print("y_jeu_III :", tuple(y_jeu_III.shape))


### 0.2 — Résumer les quatre jeux par un même calcul vectorisé · 7 min

**Objectif.** Écrire un calcul statistique qui traite les quatre jeux simultanément, en agrégeant sur le bon axe, et comprendre ce qu’un tel résumé conserve de l’information initiale.

**Ce que vous devez écrire.** La fonction `summarize_anscombe(data)` reçoit un tenseur de forme `(J, N, 2)` et renvoie une matrice de forme `(J, 7)` dont la ligne $j$ contient, dans cet ordre, la moyenne $\bar x_j$, la moyenne $\bar y_j$, la variance d’échantillon $s^2_{x,j}$, la variance $s^2_{y,j}$, la corrélation de Pearson $r_j$, la pente $a_j$ et l’ordonnée à l’origine $b_j$ de la droite des moindres carrés. Les variances utilisent le dénominateur $n-1$, et les deux derniers coefficients valent

$$a_j=\frac{\operatorname{cov}(x_j,y_j)}{s^2_{x,j}},\qquad b_j=\bar y_j-a_j\,\bar x_j .$$

**Marche à suivre.** Isolez d’abord les deux variables, ce qui donne deux tenseurs de forme `(J, N)`. Calculez les moyennes en agrégeant l’axe des observations, c’est-à-dire `dim=1`, ce qui produit des vecteurs de forme `(J,)`. Pour centrer, il faut soustraire à un tenseur `(J, N)` un vecteur `(J,)` : la diffusion ne le fait pas automatiquement, car elle aligne les axes par la droite ; ajoutez donc une dimension singleton, par exemple avec `mean_x[:, None]`, afin d’obtenir `(J, 1)`. Les variances et la covariance s’obtiennent ensuite en sommant sur `dim=1` les carrés et les produits des écarts, puis en divisant par $n-1$. Terminez par la corrélation, la pente et l’ordonnée à l’origine, qui se déduisent des quantités précédentes, et assemblez les sept vecteurs `(J,)` en une matrice `(J, 7)` avec `torch.stack(..., dim=1)`.

Une boucle sur les quatre jeux resterait acceptable, mais le calcul vectorisé rend le rôle des dimensions visible, et c’est précisément ce que l’exercice cherche à établir.

**Vérification.** La cellule suivante contrôle la forme `(4, 7)` et la finitude des valeurs. La cellule d’affichage ajoute trois contrôles numériques : les moyennes en $x$ valent exactement 9, les moyennes en $y$ avoisinent 7,5 et les quatre pentes avoisinent 0,5. Ces tolérances sont volontairement lâches, les données publiées étant arrondies.

**Erreurs fréquentes.** Agréger sur `dim=0`, ce qui mélange les quatre jeux au lieu de les résumer séparément ; utiliser `torch.var` sans se demander si son comportement par défaut correspond bien au dénominateur $n-1$ demandé ; empiler avec `dim=0` et obtenir une matrice transposée de forme `(7, J)` ; oublier la dimension singleton lors du centrage et recevoir une erreur de diffusion.

**À expliquer.** Avant d’aller plus loin, formulez une conclusion provisoire à partir du seul tableau numérique : que diriez-vous de la ressemblance entre les quatre jeux si vous n’aviez que ces sept colonnes ?


In [ ]:
def summarize_anscombe(data):
    """Retourne [mean_x, mean_y, var_x, var_y, corr, slope, intercept] pour chaque jeu."""
    # TODO 0.2 — compléter le calcul vectorisé sur l'axe des observations (dim=1).
    #
    # Étape 1 : isoler les variables.
    #   x, y de forme (J, N)  ->  indexer l'axe 2
    # Étape 2 : moyennes sur l'axe des observations.
    #   mean_x, mean_y de forme (J,)
    # Étape 3 : écarts centrés.
    #   x - mean_x[:, None] pour retrouver la forme (J, N)
    # Étape 4 : variances et covariance, dénominateur (n - 1), somme sur dim=1.
    # Étape 5 : corrélation = cov / sqrt(var_x * var_y) ; pente = cov / var_x ;
    #           intercept = mean_y - pente * mean_x
    # Étape 6 : assembler les sept vecteurs (J,) en une matrice (J, 7).
    #   torch.stack([...], dim=1)
    raise NotImplementedError("Compléter summarize_anscombe")

anscombe_stats = summarize_anscombe(ANSCOMBE)
assert anscombe_stats.shape == (4, 7)
assert torch.isfinite(anscombe_stats).all()


In [ ]:
columns = ("moy. x", "moy. y", "var. x", "var. y", "corr.", "pente", "intercept")
print("jeu | " + " | ".join(f"{name:>9s}" for name in columns))
print("-" * 88)
for label, row in zip(ANSCOMBE_LABELS, anscombe_stats):
    print(f" {label:>2s} | " + " | ".join(f"{value.item():9.4f}" for value in row))

# Les nombres ne sont pas exactement identiques à cause de l'arrondi des données originales,
# mais ils sont suffisamment proches pour conduire au même résumé verbal.
assert torch.max(torch.abs(anscombe_stats[:, 0] - 9.0)) < 1e-12
assert torch.max(torch.abs(anscombe_stats[:, 1] - 7.5)) < 1e-3
assert torch.max(torch.abs(anscombe_stats[:, 5] - 0.5)) < 5e-4


### 0.3 — Représenter sur les mêmes axes · 6 min

Le tableau que vous venez d’obtenir suggère quatre relations presque interchangeables : mêmes moyennes, mêmes variances, corrélations proches et quasiment la même droite de régression. Nous allons maintenant confronter ce résumé aux observations elles-mêmes.

**Avant d’exécuter la cellule**, prenez trente secondes et écrivez la réponse à cette question : à quoi ressemblent, selon vous, les quatre nuages de points ? La valeur pédagogique de ce prélude tient entièrement à l’écart entre cette prédiction et la figure. Si vous exécutez d’abord et réfléchissez ensuite, l’exercice perd son objet.

Les quatre panneaux partagent exactement les mêmes limites d’axes, précaution nécessaire car des échelles différentes pourraient créer ou masquer visuellement des contrastes. La droite superposée est celle que vous venez de calculer pour chaque jeu ; elle n’est pas réajustée pour « mieux suivre » la forme observée. Les points sont numérotés afin que vous puissiez désigner une observation précise dans la discussion qui suit.


In [ ]:
def plot_anscombe(data, stats):
    fig, axes = plt.subplots(2, 2, figsize=(9, 7), sharex=True, sharey=True,
                             constrained_layout=True)
    x_line = torch.linspace(3.0, 20.0, 200, dtype=data.dtype)

    for j, (label, ax) in enumerate(zip(ANSCOMBE_LABELS, axes.flat)):
        x = data[j, :, 0]
        y = data[j, :, 1]
        slope, intercept = stats[j, 5], stats[j, 6]
        ax.scatter(x.numpy(), y.numpy(), s=42, edgecolor="white", linewidth=0.8)
        ax.plot(x_line.numpy(), (slope * x_line + intercept).numpy(), linewidth=1.6)
        for i, (xi, yi) in enumerate(zip(x, y)):
            ax.annotate(str(i), (xi.item(), yi.item()), xytext=(4, 3),
                        textcoords="offset points", fontsize=7, alpha=0.7)
        ax.set_title(f"Jeu {label} — r = {stats[j, 4].item():.3f}")
        ax.set_xlim(3, 20)
        ax.set_ylim(2, 14)
        ax.grid(alpha=0.25)
        ax.set_xlabel("x")
        ax.set_ylabel("y")

    fig.suptitle("Quartet d’Anscombe — mêmes résumés, structures différentes", fontsize=13)
    export_figure(fig, "anscombe_quartet")
    plt.show()
    return fig

anscombe_figure = plot_anscombe(ANSCOMBE, anscombe_stats)


In [ ]:
# Visualisation interactive facultative : survolez les points, zoomez et comparez les panneaux.
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
except ImportError:
    print("Plotly n'est pas installé : la figure Matplotlib suffit pour poursuivre.")
else:
    fig_interactive = make_subplots(
        rows=2, cols=2,
        subplot_titles=[f"Jeu {label}" for label in ANSCOMBE_LABELS],
        shared_xaxes=True, shared_yaxes=True,
        horizontal_spacing=0.09, vertical_spacing=0.12,
    )
    x_line_np = np.linspace(3.0, 20.0, 200)
    for j, label in enumerate(ANSCOMBE_LABELS):
        row, col = divmod(j, 2)
        row, col = row + 1, col + 1
        x_np = ANSCOMBE[j, :, 0].numpy()
        y_np = ANSCOMBE[j, :, 1].numpy()
        slope = anscombe_stats[j, 5].item()
        intercept = anscombe_stats[j, 6].item()
        fig_interactive.add_trace(
            go.Scatter(
                x=x_np, y=y_np, mode="markers+text",
                text=[str(i) for i in range(len(x_np))], textposition="top center",
                customdata=np.arange(len(x_np)),
                hovertemplate="point %{customdata}<br>x=%{x:.2f}<br>y=%{y:.2f}<extra></extra>",
                name=f"Jeu {label}", showlegend=False,
            ), row=row, col=col,
        )
        fig_interactive.add_trace(
            go.Scatter(
                x=x_line_np, y=slope * x_line_np + intercept,
                mode="lines", hoverinfo="skip", showlegend=False,
            ), row=row, col=col,
        )
    fig_interactive.update_xaxes(range=[3, 20], title_text="x")
    fig_interactive.update_yaxes(range=[2, 14], title_text="y")
    fig_interactive.update_layout(
        height=650, width=900,
        title="Quartet d’Anscombe — exploration interactive",
        template="plotly_white",
    )
    fig_interactive.show()


### 0.4 — Décrire avant d’expliquer · 3 min

**Objectif.** Nommer une structure observée, plutôt que la qualifier.

**Consigne.** Rédigez une phrase par jeu, en vous interdisant les mots « bizarre » et « normal », qui décrivent votre réaction et non les données. Une description utile suit la forme : *ce que l’on voit* — *la statistique qui ne le montre pas* — *le contrôle que l’on ferait ensuite*. Par exemple, pour un jeu fictif : « les ordonnées croissent puis décroissent ; la corrélation linéaire reste proche de zéro alors que la relation est forte ; on examinerait les résidus de la droite en fonction de $x$. »

Traitez les quatre cas dans cet ordre. Pour le **jeu I**, demandez-vous quelle relation simple paraît raisonnable et quelle dispersion subsiste autour d’elle. Pour le **jeu II**, identifiez la propriété de la relation qui reste invisible dans la corrélation et dans la droite. Pour le **jeu III**, repérez le point qui attire l’attention et proposez une manière d’étudier son influence sans le supprimer arbitrairement. Pour le **jeu IV**, imaginez ce qu’il adviendrait de la variance de $x$ et de la régression si le point d’abscisse 19 était absent.

Concluez enfin par une phrase de portée générale : quelle règle méthodologique transporteriez-vous vers des jeux de données de grande dimension, où l’on ne peut plus tout afficher directement ?

**Transition.** Le quartet ne montre pas que les statistiques sont inutiles ; il montre qu’un résumé n’épuise pas la structure des données. Dans le TP A, nous conserverons cette exigence d’observation tout en construisant un protocole adapté à un échantillon plus grand, que l’on ne pourra plus inspecter point par point.


<a id="tp-a"></a>
## TP A — Des observations à un protocole expérimental · 20 min

Le quartet d’Anscombe vient de montrer qu’un tableau de statistiques descriptives ne suffit pas à caractériser la structure d’un jeu de données. Nous passons maintenant d’une petite collection que l’on peut inspecter entièrement à un échantillon plus grand, destiné à l’apprentissage. La difficulté n’est plus seulement de **voir** les données : il faut organiser une expérience permettant de distinguer ce que le modèle ajuste, ce qui sert à le choisir et ce qui restera réservé à l’évaluation finale.

**Ce que vous produirez dans cette partie.** Une fonction affine vectorisée, écrite avec la convention de dimensions du cours, et une transformation de normalisation dont les paramètres sont estimés sur le seul ensemble d’entraînement. Ces deux objets serviront dans toutes les parties suivantes.

### Le problème de classification

Les entrées sont deux coordonnées tirées uniformément dans $[-1.6,1.6]^2$. La classe vaut 1 lorsque leur produit est positif, puis chaque étiquette est inversée avec une probabilité de 8 %. Il s’agit d’une géométrie de type XOR, avec échange des noms de classes : les coordonnées de même signe portent ici l’étiquette 1.

Le bruit d’étiquette joue un rôle essentiel dans tout ce qui suivra. Même un oracle connaissant parfaitement la règle géométrique ne peut prédire les inversions aléatoires ; son exactitude moyenne plafonne donc à 92 %, et un échantillon fini ne contiendra pas nécessairement exactement 8 % d’étiquettes inversées. Une erreur observée peut ainsi provenir du bruit, d’une famille de modèles insuffisante, d’un apprentissage imparfait ou de la simple fluctuation d’échantillonnage ; ces quatre causes devront rester distinctes dans vos conclusions.

**Protocole fixé avant l’apprentissage.** Neuf cents observations, réparties en 540 exemples d’entraînement, 180 de validation et 180 de test. Le partage aléatoire est acceptable ici parce que les observations sont indépendantes et issues de la même distribution par construction ; pour des séries temporelles, plusieurs images d’un même objet ou des mesures groupées par sujet, ce même découpage créerait une fuite et donnerait une vision excessivement favorable de la généralisation. Notez que le jeu de test est immédiatement mis de côté dans la variable `TEST_SCELLE` et ne sera rouvert qu’à la fin du TP D.


In [ ]:
def make_xor(n=900, seed=SEED, flip_probability=0.08):
    g = torch.Generator().manual_seed(seed)
    x = 3.2 * torch.rand(n, 2, generator=g) - 1.6
    y_clean = (x[:, 0] * x[:, 1] > 0).long()
    flips = torch.rand(n, generator=g) < flip_probability
    y = torch.logical_xor(y_clean.bool(), flips).long()
    return x, y

X_all, y_all = make_xor()
g_split = torch.Generator().manual_seed(SEED + 1)
indices = torch.randperm(len(X_all), generator=g_split)
id_train, id_val, id_test = indices[:540], indices[540:720], indices[720:]
X_train_raw, y_train = X_all[id_train], y_all[id_train]
X_val_raw, y_val = X_all[id_val], y_all[id_val]
# Le test est scellé : aucun score, graphique ou réglage avant la fin de D.
TEST_SCELLE = (X_all[id_test].clone(), y_all[id_test].clone())
del X_all, y_all
assert set(id_train.tolist()).isdisjoint(id_val.tolist())
assert set(id_train.tolist()).isdisjoint(id_test.tolist())
assert set(id_val.tolist()).isdisjoint(id_test.tolist())
print("Apprentissage / validation / test :", len(id_train), len(id_val), len(id_test))
print("Équilibre des classes (train seulement) :", torch.bincount(y_train).tolist())

fig, ax = plt.subplots(figsize=(5.2, 4.1))
ax.scatter(X_train_raw[:, 0], X_train_raw[:, 1], c=y_train, cmap="coolwarm", s=12, alpha=.7)
ax.set(xlabel="$x_1$", ylabel="$x_2$", title="Échantillon d'apprentissage : XOR bruité", aspect="equal")
export_figure(fig, "classification_data")
plt.show()


### A1 — Lire une couche affine dans ses dimensions · 5 min

**Objectif.** Écrire une couche affine sur un mini-lot entier, sans boucle sur les observations, et savoir justifier chaque dimension du calcul.

**Ce que vous devez écrire.** La fonction `affine(x, w, b)` reçoit un mini-lot $X$ de forme $(N,d)$, une matrice de poids $W$ de forme $(d,C)$ et un biais $b$ de forme $(C,)$, et renvoie les scores

$$Z=XW+b,$$

de forme $(N,C)$. Une seule ligne suffit ; c’est sa justification qui compte.

**Marche à suivre.** Vérifiez d’abord que le produit matriciel est licite en alignant les dimensions intérieures : $(N,d)$ multiplié par $(d,C)$ donne bien $(N,C)$. L’opérateur de produit matriciel est `@`, à ne pas confondre avec `*`, qui effectue une multiplication terme à terme. Le biais, de forme $(C,)$, est ensuite ajouté : la diffusion des dimensions aligne les axes par la droite, complète implicitement $b$ en une ligne $(1,C)$ et la réplique sur les $N$ lignes. Écrivez cette chaîne de formes en commentaire avant de coder, puis vérifiez que le résultat la respecte.

**Vérification.** La cellule de test emploie un exemple numérique de petite taille avec $N=3$, $d=2$ et $C=3$. Elle contrôle la forme du résultat, puis compare la première ligne à la valeur attendue $(5{,}5;\,1{,}5;\,0)$. Vous pouvez la recalculer à la main en une minute, et il est recommandé de le faire au moins une fois dans la séance. La dernière ligne affiche la forme de `nn.Linear(2, 3).weight` : observez qu’elle est **transposée** par rapport à notre convention.

**Erreurs fréquentes.** Écrire `w @ x`, ce qui échoue sur les dimensions ; utiliser `*` au lieu de `@` et obtenir une erreur de diffusion, ou pire, un résultat de forme plausible mais faux ; ajouter le biais après une transposition et le diffuser sur le mauvais axe.

**À expliquer.** Pourquoi le même biais est-il ajouté à chacune des $N$ lignes, et que signifierait un biais qui différerait d’une observation à l’autre ? Quelle forme PyTorch utilise-t-il pour `nn.Linear(d, C).weight`, et pourquoi cette convention interne n’est-elle pas une inconséquence mais un choix lié à l’écriture $y = xA^\top + b$ ? Savoir passer d’une convention à l’autre évite un grand nombre d’erreurs silencieuses.


In [ ]:
def affine(x, w, b):
    # TODO A1 : renvoyer les scores de toutes les observations, sans boucle.
    #   x : (N, d)   w : (d, C)   b : (C,)   ->   sortie attendue : (N, C)
    #   produit matriciel : @      puis addition du biais, diffusée sur les N lignes
    raise NotImplementedError("A1 : produit matriciel et diffusion du biais")


In [ ]:
a = torch.tensor([[1., 2.], [3., 4.], [-1., 2.]])
w = torch.tensor([[1., 0., -1.], [2., 1., 0.]])
b = torch.tensor([0.5, -0.5, 1.])
z = affine(a, w, b)
assert z.shape == (3, 3)
assert torch.allclose(z[0], torch.tensor([5.5, 1.5, 0.]))
print("Forme des scores :", tuple(z.shape))
print("Stockage nn.Linear(2, 3).weight :", tuple(nn.Linear(2, 3).weight.shape))


### A2 — Transformer les données sans consulter l’avenir · 10 min

**Objectif.** Comprendre qu’une transformation de données possède elle-même des paramètres, et que ces paramètres appartiennent à l’apprentissage.

**Ce que vous devez écrire.** La fonction `fit_standardizer(x_train)` reçoit le tenseur d’entraînement, de forme $(N,d)$, et renvoie le couple `(mean, std)` de deux tenseurs de forme $(1,d)$, calculés **colonne par colonne sur ce seul ensemble**. Les cellules suivantes les appliquent sans modification à l’entraînement puis à la validation, et le TP D les réutilisera pour le test.

**Marche à suivre.** Agrégez sur l’axe des observations, donc `dim=0`, et conservez la dimension singleton avec `keepdim=True` : c’est ce qui donne la forme $(1,d)$ et rend la diffusion visible dans le code plutôt que devinée. Pour l’écart-type, utilisez `unbiased=False`, car nous décrivons ici l’échantillon d’entraînement et non une population dont il faudrait estimer la variance sans biais. Appliquez enfin `clamp_min(1e-6)` au résultat, afin de traiter explicitement le cas d’une coordonnée constante : une division par zéro produirait des `inf` qui se propageraient silencieusement jusqu’à la perte.

**Vérification.** Les assertions contrôlent que les deux tenseurs ont bien la forme $(1,2)$, que les données d’entraînement transformées ont une moyenne nulle et un écart-type unité, et que les cibles sont bien de type `torch.long`. Les deux moyennes affichées, celle de l’entraînement puis celle de la validation, sont à lire attentivement : la première est nulle à la précision numérique près, la seconde ne l’est pas.

**Erreurs fréquentes.** Oublier `keepdim=True` et obtenir des tenseurs $(d,)$ qui fonctionneront ici par chance mais masqueront le mécanisme ; normaliser la validation avec sa propre moyenne, ce qui constitue une fuite ; recalculer les statistiques à chaque appel du modèle, ce qui rend la procédure non reproductible.

**À expliquer.** Il est normal que les données de validation transformées n’aient ni une moyenne exactement nulle ni un écart-type égal à un, puisqu’elles constituent un autre échantillon. Leur retirer leur propre moyenne utiliserait une information indisponible au moment du déploiement et modifierait la procédure que l’on prétend évaluer. Formulez cette idée dans vos propres termes : où, précisément, se situerait la fuite ?


In [ ]:
def fit_standardizer(x_train):
    # TODO A2 : calculer deux tenseurs de forme (1, d), sur le train uniquement.
    #   moyenne : agréger sur dim=0 en conservant la dimension (keepdim=True)
    #   écart-type : même axe, unbiased=False, puis .clamp_min(1e-6)
    #   renvoyer le couple (mean, std)
    raise NotImplementedError("A2 : moyenne et écart-type sur le train uniquement")

mean_train, std_train = fit_standardizer(X_train_raw)
X_train = (X_train_raw - mean_train) / std_train
X_val = (X_val_raw - mean_train) / std_train


In [ ]:
assert mean_train.shape == std_train.shape == (1, 2)
assert torch.allclose(X_train.mean(0), torch.zeros(2), atol=1e-6)
assert torch.allclose(X_train.std(0, unbiased=False), torch.ones(2), atol=1e-6)
assert y_train.dtype == torch.long
print("Moyenne train après transformation :", X_train.mean(0).tolist())
print("Moyenne validation après transformation :", X_val.mean(0).tolist())
print("Entrées :", X_train.dtype, tuple(X_train.shape), "| cibles :", y_train.dtype, tuple(y_train.shape))


### Bilan du TP A — formuler ce que le code a établi

Rédigez deux ou trois phrases par question. Une notation correcte ne suffit pas : reliez chaque calcul à la logique expérimentale, car ce sont ces liens, et non les lignes de code, qui seront réutilisés dans les parties suivantes.

La première question porte sur les dimensions : quelles sont celles de $X$, $W$, $b$ et $Z$, et par quel mécanisme le biais est-il ajouté aux $N$ lignes ? La deuxième porte sur la transformation : pourquoi les statistiques de normalisation appartiennent-elles à l’apprentissage, et à quel endroit précis une fuite de données apparaîtrait-elle si l’on s’y prenait autrement ? La troisième porte sur le découpage : quelle hypothèse autorise ici un partage aléatoire, et pour quels types de données faudrait-il la remettre en cause ?


<a id="tp-b"></a>
## TP B — De la frontière affine à la représentation apprise · 30 min

Le protocole étant fixé, nous pouvons poser une question d’une autre nature : **la famille de fonctions choisie est-elle capable de représenter la structure observée ?** Aucune optimisation, si soignée soit-elle, ne peut trouver une fonction qui n’appartient pas à cette famille. La comparaison d’un modèle affine et d’un réseau multicouche isolera cette difficulté de représentation, en la séparant de la question de l’optimisation, que nous traiterons seulement au TP C.

### B1 — Deux logits pour une décision binaire · 8 min

**Objectif.** Construire un perceptron multicouche minimal et comprendre pourquoi sa dernière couche ne doit pas être suivie d’un softmax.

**Ce que vous devez écrire.** La fonction `make_mlp(width=32)` renvoie un module PyTorch réalisant l’enchaînement $2\to\text{width}\to\text{width}\to 2$, c’est-à-dire trois couches affines séparées par deux activations `Tanh`, et **aucune activation en sortie**. Appliqué à un tenseur $(N,2)$, le module doit renvoyer un tenseur $(N,2)$.

**Marche à suivre.** Le conteneur `nn.Sequential` applique ses modules dans l’ordre où ils sont donnés ; il suffit donc de les énumérer. Comptez-les avant d’écrire : une couche `nn.Linear(2, width)`, une activation, une couche `nn.Linear(width, width)`, une activation, puis une couche `nn.Linear(width, 2)`, soit cinq modules. Vérifiez que les dimensions se raccordent deux à deux, la sortie de chaque couche devant être l’entrée de la suivante.

**Ce qu’il faut comprendre avant de coder.** Un classifieur à deux classes produit ici deux scores réels, appelés **logits**. Le softmax les transformerait en probabilités, mais `CrossEntropyLoss` reçoit directement les logits et effectue elle-même une évaluation numériquement stable de la log-softmax et de la perte ; ajouter un softmax dans le modèle appliquerait donc l’opération deux fois. Les cibles sont les indices de classes 0 ou 1, de forme $(N,)$ et de type `torch.long`, et non des vecteurs indicateurs. Quant à `Tanh`, elle n’est pas présentée comme universellement supérieure : elle convient à ce petit problème et sera réutilisée dans le PINN du TP E, où la régularité de ses dérivées joue un rôle explicite.

**Vérification.** La cellule de contrôle applique le modèle à sept observations et vérifie que la sortie a la forme $(7,2)$, puis parcourt les sous-modules pour s’assurer qu’aucun `nn.Softmax` n’a été inséré. Elle affiche enfin l’architecture et le nombre de paramètres entraînables : lisez ce nombre et retrouvez-le par le calcul, il vaut $2\cdot32+32$ pour la première couche, et ainsi de suite.

**Erreurs fréquentes.** Placer une activation après la dernière couche linéaire, ce qui écrase les logits ; oublier une activation intermédiaire, auquel cas la composition de deux couches affines reste affine et le réseau perd tout intérêt ; raccorder mal les largeurs et déclencher une erreur de dimensions dès le premier appel.


In [ ]:
def make_mlp(width=32):
    # TODO B1 : architecture 2 -> width -> width -> 2
    #   nn.Sequential attend les modules dans l'ordre d'application :
    #   Linear(2, width), Tanh, Linear(width, width), Tanh, Linear(width, 2)
    #   aucune activation après la dernière couche : la perte reçoit des logits
    raise NotImplementedError("B1 : architecture 2 → width → width → 2")


In [ ]:
probe = make_mlp()
assert probe(X_train[:7]).shape == (7, 2)
assert not any(isinstance(m, nn.Softmax) for m in probe.modules())
print(probe)
print("Paramètres entraînables :", sum(p.numel() for p in probe.parameters()))


### Le moteur d’entraînement fourni — comprendre son contrat avant de l’utiliser

La fonction `train_classifier` est fournie afin que cette première comparaison porte sur les **familles de fonctions** et non sur l’écriture de la boucle d’optimisation, que vous écrirez vous-même au TP C. Prenez néanmoins deux minutes pour lire son contrat, car vous l’utiliserez quatre fois dans la séance.

Elle ajuste les paramètres à partir des mini-lots de l’ensemble d’entraînement, recalcule à chaque époque les pertes complètes en mode évaluation, conserve une **copie indépendante** des poids correspondant à la meilleure perte de validation, puis restaure cette copie avant de rendre le modèle. Elle renvoie un dictionnaire contenant le modèle restauré, l’historique des quatre courbes, la meilleure perte de validation, l’époque correspondante, le budget d’époques et la valeur de `weight_decay` employée ; le TP D lira précisément ces clés. Elle ne reçoit aucun jeu de test, et cette absence n’est pas un détail d’interface : elle protège structurellement le rôle du test dans l’expérience.

Deux mécanismes distincts apparaissent dans son code et seront réexaminés au TP C. Les appels `model.train()` et `model.eval()` règlent le comportement de certaines couches, tandis que `torch.no_grad()` contrôle l’enregistrement des opérations nécessaires au calcul des gradients ; ils répondent à deux questions différentes et ne se remplacent pas l’un l’autre. Le MLP de cette partie ne contient pas de dropout, mais nous conservons une fonction d’évaluation correcte et réutilisable.

**Décision annoncée avant les résultats.** Le TP D comparera quatre candidats : le modèle linéaire, le MLP court de cette partie, puis deux MLP entraînés plus longtemps, avec et sans `weight_decay`. La sélection se fera exclusivement à partir de la perte de validation. Annoncer ce protocole maintenant, avant d’avoir vu le moindre résultat, fait partie de la méthode : c’est ce qui distingue une comparaison d’une justification rétrospective.


In [ ]:
loss_fn = nn.CrossEntropyLoss()

@torch.no_grad()
def evaluate_classifier(model, x, y):
    model.eval()
    logits = model(x)
    return {"loss": float(loss_fn(logits, y)),
            "accuracy": float((logits.argmax(dim=1) == y).float().mean())}

def train_classifier(model, x_train, y_train, x_val, y_val,
                     epochs=180, lr=0.01, weight_decay=0.0, seed=SEED + 2):
    model = model.to(DEVICE)
    loader = DataLoader(TensorDataset(x_train, y_train), batch_size=64,
                        shuffle=True, generator=torch.Generator().manual_seed(seed))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = {"train_loss": [], "val_loss": [], "train_accuracy": [], "val_accuracy": []}
    best_loss, best_epoch, best_state = float("inf"), 0, None
    for epoch in range(1, epochs + 1):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
        train_metrics = evaluate_classifier(model, x_train, y_train)
        val_metrics = evaluate_classifier(model, x_val, y_val)
        for split, metrics in [("train", train_metrics), ("val", val_metrics)]:
            for key, value in metrics.items():
                history[f"{split}_{key}"].append(value)
        if val_metrics["loss"] < best_loss:
            best_loss, best_epoch = val_metrics["loss"], epoch
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    model.eval()
    return {"model": model, "history": history, "best_val_loss": best_loss,
            "best_epoch": best_epoch, "epochs": epochs,
            "weight_decay": weight_decay}

def show_histories(experiments, figure_name="classification_learning"):
    fig, axes = plt.subplots(1, 2, figsize=(10.8, 3.5))
    for name, result in experiments.items():
        h = result["history"]
        epochs = np.arange(1, len(h["train_loss"]) + 1)
        line = axes[0].plot(epochs, h["train_loss"], label=name + " train")[0]
        axes[0].plot(epochs, h["val_loss"], "--", color=line.get_color(), label=name + " val")
        axes[1].plot(epochs, h["val_accuracy"], label=name)
    axes[0].set(xlabel="Époque", ylabel="Entropie croisée", title="Perte : apprentissage / validation")
    axes[1].set(xlabel="Époque", ylabel="Exactitude", title="Validation seulement", ylim=(.35, 1.0))
    for ax in axes:
        ax.legend(fontsize=8)
        ax.grid(alpha=.2)
    fig.tight_layout()
    export_figure(fig, figure_name)
    plt.show()

def show_boundaries(experiments):
    grid_axis = torch.linspace(-1.8, 1.8, 130)
    gx, gy = torch.meshgrid(grid_axis, grid_axis, indexing="xy")
    grid_raw = torch.stack((gx.ravel(), gy.ravel()), dim=1)
    grid = (grid_raw - mean_train) / std_train
    fig, axes = plt.subplots(1, len(experiments), figsize=(5.1 * len(experiments), 4.0), squeeze=False)
    for ax, (name, result) in zip(axes[0], experiments.items()):
        model = result["model"]
        model.eval()
        with torch.no_grad():
            probability = model(grid).softmax(dim=1)[:, 1].reshape(gx.shape)
        im = ax.contourf(gx, gy, probability, levels=np.linspace(0, 1, 15), cmap="coolwarm", vmin=0, vmax=1)
        ax.contour(gx, gy, probability, levels=[.5], colors="black", linewidths=1)
        ax.scatter(X_val_raw[:, 0], X_val_raw[:, 1], c=y_val, cmap="coolwarm", s=14, edgecolors="white", linewidths=.35)
        ax.set(title=name + " — points de validation", xlabel="$x_1$", ylabel="$x_2$", aspect="equal")
    fig.colorbar(im, ax=axes.ravel().tolist(), label=r"$p_\theta(y=1\mid x)$", shrink=.8)
    export_figure(fig, "classification_boundaries")
    plt.show()


### B2 — Prévoir la géométrie, entraîner, puis expliquer · 17 min

**Objectif.** Distinguer un défaut de représentation d’un défaut d’optimisation, à partir d’indices observables.

**Avant d’exécuter.** Dessinez sur une feuille la frontière que vous attendez pour chacun des deux modèles, et notez l’exactitude de validation que vous anticipez. Demandez-vous ensuite quel résultat permettrait de trancher entre les deux explications possibles d’une mauvaise performance : une famille de fonctions trop pauvre, ou une descente de gradient défaillante. Cette prédiction écrite est la partie la plus utile de l’exercice ; une frontière observée n’instruit que celui qui en attendait une autre.

**Grille de lecture des figures.** Lorsque la cellule a terminé, examinez quatre choses dans cet ordre. D’abord les deux pertes, d’entraînement et de validation, et l’écart qui les sépare. Ensuite la forme de la frontière de décision, qui est ici une information scientifique à part entière et non une illustration. Puis la localisation des erreurs : sont-elles dispersées, ou concentrées dans certaines régions du plan ? Enfin l’époque retenue par la validation, qui vous dit à quel moment le modèle restauré a été figé. Les figures utilisent en effet les poids de la meilleure époque de validation, et non ceux de la dernière.

**Interprétation attendue.** Une mauvaise exactitude du modèle affine n’indique pas nécessairement que l’optimisation fonctionne mal ; elle peut simplement révéler que sa frontière est trop contrainte pour la géométrie du problème. Sachez formuler cette distinction avant de passer au TP C, car toute la suite de la séance en dépend. Rappelez-vous par ailleurs le plafond de 92 % imposé par le bruit d’étiquette : une exactitude de validation proche de cette valeur ne laisse pas de marge de progrès significative.


In [ ]:
torch.manual_seed(SEED + 10)
linear = train_classifier(nn.Linear(2, 2), X_train, y_train, X_val, y_val)
torch.manual_seed(SEED + 10)
mlp = train_classifier(make_mlp(), X_train, y_train, X_val, y_val)
experiments_B = {"Linéaire": linear, "MLP": mlp}
for name, result in experiments_B.items():
    metrics = evaluate_classifier(result["model"], X_val, y_val)
    print(f"{name:10s} | meilleure époque {result['best_epoch']:3d} | "
          f"perte val {metrics['loss']:.4f} | exactitude val {metrics['accuracy']:.3f}")
show_histories(experiments_B)
show_boundaries(experiments_B)


### Bilan du TP B — distinguer représentation et optimisation

Répondez en deux ou trois phrases par question, en vous appuyant sur les figures que vous venez d’obtenir.

Pourquoi une frontière affine ne peut-elle pas séparer exactement les quatre quadrants alternés, et quel argument géométrique le montre en une ligne ? Pourquoi l’application d’un softmax avant `CrossEntropyLoss` serait-elle à la fois redondante et numériquement moins sûre ? Quels indices, observables dans une expérience, permettraient de distinguer un modèle insuffisamment expressif d’un entraînement défaillant — et quel test simple pourriez-vous conduire pour trancher ? Que garantit enfin un théorème d’approximation universelle, et que ne garantit-il ni sur les données disponibles, ni sur l’algorithme d’apprentissage ?


<a id="tp-c"></a>
## TP C — Du critère numérique à la modification des paramètres · 30 min

Nous disposons maintenant d’une famille de fonctions et d’une perte. Il reste à comprendre comment l’erreur finale est attribuée à chacun des paramètres. Cette partie ouvre la boîte de la rétropropagation : nous commencerons par une dérivation matricielle faite à la main, nous la confronterons à `autograd`, puis nous écrirons une époque complète d’apprentissage.

### C1 — Dériver, vérifier, puis automatiser · 12 min

**Objectif.** Retrouver par le calcul les gradients qu’`autograd` produit automatiquement, et se convaincre que l’autodifférentiation n’est pas une boîte noire.

**Le résultat de cours.** Pour $Z=XW+b$, $P=\operatorname{softmax}(Z)$ et une matrice indicatrice des cibles $Y$, l’entropie croisée moyenne vérifie

$$
\frac{\partial L}{\partial Z}=\frac{P-Y}{N},\qquad
\frac{\partial L}{\partial W}=X^\top\frac{P-Y}{N},\qquad
\frac{\partial L}{\partial b}=\sum_{i=1}^N\frac{P_i-Y_i}{N}.
$$

**Ce que vous devez écrire.** Dans la fonction `manual_ce_grad`, la quantité $\partial L/\partial Z$, nommée `dz`, est déjà calculée pour vous et possède la forme $(N,C)$. Il vous reste à en déduire `dw`, de forme $(d,C)$, et `db`, de forme $(C,)$, puis à renvoyer le couple `(dw, db)` dans cet ordre.

**Marche à suivre.** Raisonnez uniquement sur les dimensions, la formule suivra. Pour obtenir un objet $(d,C)$ à partir de $X$ de forme $(N,d)$ et de `dz` de forme $(N,C)$, il faut faire disparaître l’axe des observations en le contractant, ce qui impose de transposer $X$ : l’expression `x.T @ dz` réalise exactement la somme sur les $N$ exemples annoncée par la formule. Pour `db`, la somme sur les exemples s’écrit directement comme une agrégation de `dz` sur `dim=0`, qui supprime cet axe et laisse la forme $(C,)$. Conservez la convention $(d,C)$ adoptée en A1 ; le passage par la convention transposée de `nn.Linear` est une source d’erreurs qu’il n’est pas utile d’introduire ici.

**Vérification.** La cellule suivante construit un petit problème en double précision, calcule les mêmes gradients par `torch.autograd.grad`, affiche les écarts maximaux et vérifie qu’ils sont négligeables. Elle appelle ensuite `gradcheck`, qui compare l’autodifférentiation à des différences finies : il s’agit d’un **contrôle local indépendant**, non d’une méthode d’apprentissage. L’accord de trois calculs obtenus par trois voies distinctes est beaucoup plus informatif que la seule observation d’une perte qui diminue. Le recours à `float64` sert ici à contrôler des dérivées et ne signifie pas que les entraînements doivent être menés en double précision.

**Erreurs fréquentes.** Écrire `dz @ x.T`, ce qui donne une forme $(N,N)$ dépourvue de sens ici ; sommer `dz` sur `dim=1`, c’est-à-dire agréger les classes au lieu des exemples ; diviser une seconde fois par $N$, alors que la division est déjà contenue dans `dz`.


In [ ]:
def manual_ce_grad(x, w, b, y):
    probabilities = (x @ w + b).softmax(dim=1)
    one_hot = F.one_hot(y, num_classes=w.shape[1]).to(x.dtype)
    dz = (probabilities - one_hot) / len(x)      # (N, C), division par N déjà faite
    # TODO C1 : calculer dw et db, puis les renvoyer dans cet ordre.
    #   dw : contracter l'axe des observations entre x (N, d) et dz (N, C)  ->  (d, C)
    #   db : sommer dz sur l'axe des observations                            ->  (C,)
    raise NotImplementedError("C1 : gradient matriciel et somme sur le mini-lot")


In [ ]:
g = torch.Generator().manual_seed(SEED + 20)
xg = torch.randn(5, 2, generator=g, dtype=torch.float64)
wg = torch.randn(2, 3, generator=g, dtype=torch.float64, requires_grad=True)
bg = torch.randn(3, generator=g, dtype=torch.float64, requires_grad=True)
yg = torch.tensor([0, 1, 2, 0, 1], dtype=torch.long)
lg = F.cross_entropy(xg @ wg + bg, yg)
dw_auto, db_auto = torch.autograd.grad(lg, (wg, bg))
dw_manual, db_manual = manual_ce_grad(xg, wg.detach(), bg.detach(), yg)
print("Écart maximal sur W :", float((dw_manual - dw_auto).abs().max()))
print("Écart maximal sur b :", float((db_manual - db_auto).abs().max()))
assert torch.allclose(dw_manual, dw_auto, atol=1e-10, rtol=1e-8)
assert torch.allclose(db_manual, db_auto, atol=1e-10, rtol=1e-8)
passed = torch.autograd.gradcheck(lambda w, b: F.cross_entropy(xg @ w + b, yg),
                                 (wg, bg), eps=1e-6, atol=1e-5, rtol=1e-3)
print("Vérification par différences finies :", passed)


### C1 bis — L’accumulation des gradients est un choix algorithmique · 3 min

**Objectif.** Constater que `.grad` accumule, et en tirer les conséquences sur la place de `zero_grad` dans un algorithme.

**Avant d’exécuter.** La cellule suivante dérive trois fois la fonction $a\mapsto a^2$ en $a=2$, sans effacer les gradients entre les deux premiers appels, puis en les effaçant avant le troisième. Écrivez les trois nombres que vous attendez avant de lancer le calcul ; l’assertion finale vous dira si votre modèle mental est correct.

**Ce qu’il faut retenir.** Dans PyTorch, `backward()` **ajoute** sa contribution au contenu de `.grad` et ne remplace pas le gradient précédent. Cette accumulation est utile lorsqu’elle est intentionnelle, par exemple pour simuler un lot plus grand que la mémoire disponible en cumulant plusieurs mini-lots, mais elle change silencieusement l’algorithme si l’on oublie de remettre les gradients à zéro. `zero_grad` n’est donc pas une opération de ménage : elle fait partie de la définition de la mise à jour. Notons enfin que nous reconstruisons ici le graphe à chaque appel, de sorte que `retain_graph=True` n’est pas nécessaire.


In [ ]:
a = torch.tensor(2., requires_grad=True)
(a * a).backward()
first = a.grad.item()
(a * a).backward()
accumulated = a.grad.item()
a.grad = None
(a * a).backward()
after_reset = a.grad.item()
print("Premier backward / second sans effacement / après effacement :", first, accumulated, after_reset)
assert (first, accumulated, after_reset) == (4., 8., 4.)


### C2 — Écrire une époque d’apprentissage complète · 10 min

**Objectif.** Écrire soi-même la boucle que le moteur du TP B exécutait, et savoir associer chaque ligne à une opération mathématique identifiée.

**Ce que vous devez écrire.** Le corps de la boucle `for xb, yb in loader:` dans la fonction `train_one_epoch`, qui parcourt les mini-lots, met à jour les paramètres et renvoie la perte moyenne de l’époque. Une contrainte technique : les deux lignes de comptabilité placées après votre code utilisent une variable nommée `loss`, que vous devez donc définir sous ce nom exact.

**Marche à suivre.** Cinq opérations se succèdent toujours dans le même ordre. Il faut d’abord effacer les gradients hérités de la mise à jour précédente, avec `optimizer.zero_grad(set_to_none=True)`, pour la raison examinée en C1 bis. Il faut ensuite calculer les logits du mini-lot en appliquant le modèle à `xb`, puis construire la perte en confrontant ces logits à `yb` au moyen de `loss_fn`, définie plus haut dans le notebook. Il faut alors propager les dérivées par `loss.backward()`, qui remplit les attributs `.grad` de tous les paramètres. Il faut enfin demander à l’optimiseur de modifier les paramètres par `optimizer.step()`, qui lit ces gradients ainsi que son propre état interne.

**Vérification.** La cellule suivante lance trente époques et affiche la première et la dernière perte moyenne, puis la performance de validation. Le résultat attendu n’est pas un score spectaculaire mais une décroissance nette et une boucle dont chaque ligne s’explique. Remarquez le message final : cette perte de mini-lots agrège des paramètres successifs, puisqu’elle est mesurée pendant que le modèle change, et diffère donc d’une perte recalculée en fin d’époque sur le modèle figé. Le modèle entraîné ici sert à vérifier la mécanique et **ne s’ajoute pas** aux quatre candidats du TP D.

**Erreurs fréquentes.** Oublier `zero_grad` et cumuler les gradients de tous les mini-lots, ce qui produit une descente étrangement instable sans lever la moindre erreur ; appeler `step()` avant `backward()`, auquel cas la mise à jour utilise des gradients périmés ; nommer la perte autrement que `loss` et déclencher un `NameError` deux lignes plus bas ; appeler `.item()` sur la perte avant `backward()`, ce qui rompt le lien avec le graphe de calcul.


In [ ]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss, total_examples = 0.0, 0
    for xb, yb in loader:
        # TODO C2 : les cinq étapes de l'apprentissage, dans l'ordre.
        #   1) effacer les gradients hérités de la mise à jour précédente
        #   2) calculer les logits du mini-lot à partir de xb
        #   3) construire la perte avec loss_fn ; elle doit s'appeler loss
        #   4) propager les dérivées
        #   5) demander la mise à jour des paramètres à l'optimiseur
        raise NotImplementedError("C2 : zéro → prédiction/perte → backward → step")
        total_loss += loss.detach().item() * len(xb)
        total_examples += len(xb)
    return total_loss / total_examples


In [ ]:
torch.manual_seed(SEED + 30)
model_C = make_mlp()
loader_C = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True,
                      generator=torch.Generator().manual_seed(SEED + 31))
optimizer_C = torch.optim.AdamW(model_C.parameters(), lr=.01, weight_decay=0.)
losses_C = [train_one_epoch(model_C, loader_C, optimizer_C) for _ in range(30)]
assert np.isfinite(losses_C).all()
print("Perte moyenne rencontrée dans les mini-lots :", round(losses_C[0], 4), "→", round(losses_C[-1], 4))
print("Validation après 30 époques :", evaluate_classifier(model_C, X_val, y_val))
print("Cette perte de mini-lots agrège des paramètres successifs ; elle diffère d'une perte recalculée en fin d'époque.")


### Bilan du TP C — séparer dérivation, état du modèle et mise à jour

Quatre questions, dont les trois premières portent sur des confusions très répandues. Quelle instruction calcule les gradients, et quelle instruction modifie les paramètres ? `model.eval()` interdit-il le calcul d’un gradient, et que change-t-il réellement ? `torch.no_grad()` désactive-t-il le dropout, et pourquoi ces deux mécanismes ne sont-ils pas interchangeables ? Pourquoi enfin pondère-t-on la perte de chaque mini-lot par `len(xb)` avant de calculer la moyenne d’une époque ?

Cette dernière question n’est pas un détail d’implémentation : gardez-la en tête pour le TP E, où la perte agrégera des résidus sur des points de collocation dont le nombre est fixé par vous.


<a id="tp-d"></a>
## TP D — De l’ajustement à une conclusion défendable · 25 min

Une faible perte d’entraînement prouve seulement que le modèle s’est adapté aux observations qui ont influencé ses paramètres. Elle ne dit ni quelle configuration il faut retenir, ni ce que cette configuration fera sur de nouvelles observations. Cette partie rend donc explicite la différence entre trois opérations que l’on confond souvent : **entraîner**, **sélectionner** et **évaluer**.

### Deux expériences contrôlées · 10 min

La cellule suivante entraîne le même MLP pendant 350 époques avec deux valeurs annoncées à l’avance pour `weight_decay`, zéro et 0,02. L’initialisation, le taux d’apprentissage, l’ordre des mini-lots et le budget restent identiques : une seule chose change, et c’est ce qui autorise à attribuer une différence observée à cette chose. AdamW applique une **décroissance découplée des poids** ; pour un optimiseur adaptatif, cette opération n’équivaut pas à l’ajout naïf d’une pénalité quadratique au gradient, comme le complément hors séance le détaille. Afin de garder le code lisible, la décroissance s’applique ici à tous les paramètres, biais compris, ce qui n’est pas l’usage le plus courant.

**Avant d’exécuter**, dessinez plusieurs scénarios plausibles pour les courbes, et non un seul : une régularisation peut améliorer la validation, rester sans effet mesurable, ou détériorer l’optimisation. Anticiper les trois cas vous évitera de lire la figure comme une confirmation. L’expérience ne comporte qu’une seule graine ; elle fournit un exemple reproductible, non une loi générale sur les MLP.


In [ ]:
experiments_D = {}
for wd in (0.0, 0.02):
    torch.manual_seed(SEED + 40)  # même initialisation dans les deux expériences
    result = train_classifier(make_mlp(), X_train, y_train, X_val, y_val,
                              epochs=350, lr=.01, weight_decay=wd, seed=SEED + 41)
    name = f"MLP long, wd={wd:g}"
    experiments_D[name] = result
    print(f"{name:24s} | meilleure époque {result['best_epoch']:3d} | perte val {result['best_val_loss']:.4f}")
show_histories(experiments_D, "regularization_learning")


### D1 — Sélectionner sur la validation et figer le bon état · 7 min

**Objectif.** Écrire une règle de sélection explicite, et comprendre que la validation participe à l’apprentissage bien qu’aucun gradient n’y soit calculé.

**Ce que vous devez écrire.** La fonction `choose_by_validation(candidates)` reçoit le dictionnaire `candidates`, dont les clés sont les noms des quatre modèles annoncés au TP B et dont chaque valeur est le dictionnaire renvoyé par `train_classifier`. Elle doit renvoyer le **nom** du candidat dont la valeur associée à la clé `best_val_loss` est la plus faible — le nom, donc une chaîne de caractères, et non le dictionnaire ni le modèle.

**Marche à suivre.** La fonction native `min` itère sur les clés d’un dictionnaire et accepte un argument `key`, qui est une fonction appliquée à chaque clé pour produire la quantité à comparer. Il vous faut donc écrire une fonction anonyme qui, à un nom, associe la meilleure perte de validation du résultat correspondant. Une boucle explicite conviendrait aussi ; l’essentiel est que le critère apparaisse en un seul endroit du code, de sorte qu’il ne puisse pas être modifié discrètement après coup.

**Vérification.** La cellule affiche le choix retenu et l’époque à laquelle les poids ont été figés. La fonction `train_classifier` a déjà restauré les poids correspondant à cette époque ; vous n’avez donc ni à recharger un état, ni à réentraîner. Ne choisissez ni le dernier état par habitude, ni une variante nouvelle qu’un résultat de test encore à venir vous inspirerait.

**Erreurs fréquentes.** Renvoyer `min(candidates.values(), ...)`, ce qui perd le nom ; comparer les pertes finales plutôt que les meilleures pertes de validation ; introduire l’exactitude dans le critère alors que le protocole annoncé portait sur la perte.

**À expliquer.** Pourquoi `best_state = model.state_dict()` sans copie profonde ne suffit-il pas toujours à figer un point de contrôle — que contient exactement le dictionnaire renvoyé, et que devient son contenu au pas suivant ? Pourquoi la validation participe-t-elle indirectement au processus d’apprentissage, alors même qu’aucun `backward()` n’est calculé sur ses observations ? Votre réponse doit faire apparaître la notion de **sélection** : un choix effectué à partir de données transmet à ces données une part de l’information dont le résultat dépend.


In [ ]:
candidates = {**experiments_B, **experiments_D}

def choose_by_validation(candidates):
    # TODO D1 : renvoyer le NOM du candidat de plus petite best_val_loss.
    #   candidates : dict {nom : résultat}, où résultat["best_val_loss"] est un float
    #   indice : min(candidates, key=...) itère sur les clés et compare la quantité choisie
    raise NotImplementedError("D1 : une sélection fondée exclusivement sur la validation")

chosen_name = choose_by_validation(candidates)
chosen_result = candidates[chosen_name]
chosen_model = chosen_result["model"]
print("Choix verrouillé :", chosen_name, "| époque", chosen_result["best_epoch"])


### D2 — Ouvrir le test une seule fois · 3 min

Le modèle étant choisi et verrouillé, nous pouvons évaluer la procédure finale sur les 180 observations réservées. Elles reçoivent la transformation déterminée sur le train au TP A ; aucune statistique n’est réestimée, ce qui est exactement la situation d’un déploiement.

La cellule est protégée par un drapeau : si le test a déjà été ouvert dans cette session, elle réaffiche le rapport initial au lieu d’en produire un nouveau. Ce garde-fou matérialise une règle de méthode. Dès lors qu’un résultat de test influence un nouveau réglage, ce jeu cesse d’être un test indépendant pour l’étude en cours ; il serait tout aussi incorrect de relancer plusieurs graines et de ne conserver que celle qui donne le nombre le plus flatteur.

**Comment lire le rapport.** L’intervalle de Wilson à 95 % décrit l’incertitude binomiale associée à l’exactitude de ce prédicteur fixé, sous l’hypothèse d’observations indépendantes. Il ne mesure ni la variabilité due à une autre initialisation, ni l’incertitude liée au choix de l’architecture, ni un éventuel décalage de distribution. Sa largeur, sur 180 observations, mérite d’être comparée aux écarts que vous avez observés entre candidats en validation : c’est souvent cette comparaison, plus que le score lui-même, qui détermine ce qu’il est raisonnable d’affirmer.


In [ ]:
def wilson_interval(k, n, z=1.959963984540054):
    p = k / n
    denominator = 1 + z*z/n
    center = (p + z*z/(2*n)) / denominator
    radius = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / denominator
    return center - radius, center + radius

if globals().get("TEST_ALREADY_OPENED", False):
    print("Test déjà ouvert : conserver le résultat initial ci-dessous, sans nouveau réglage.")
    print(final_report)
else:
    X_test_raw, y_test = TEST_SCELLE
    X_test = (X_test_raw - mean_train) / std_train
    chosen_model.eval()
    with torch.no_grad():
        test_logits = chosen_model(X_test)
        test_loss = float(loss_fn(test_logits, y_test))
        correct = int((test_logits.argmax(1) == y_test).sum())
    low, high = wilson_interval(correct, len(y_test))
    final_report = {"model": chosen_name, "best_epoch": chosen_result["best_epoch"],
                    "test_loss": test_loss, "test_accuracy": correct / len(y_test),
                    "wilson_95": (low, high), "n_test": len(y_test)}
    TEST_ALREADY_OPENED = True
    print(final_report)


### Bilan du TP D — limiter la portée de ce que l’on affirme

Quelle époque aurait été retenue si l’on avait minimisé uniquement la perte d’entraînement, et à quelle question différente cette règle répondrait-elle ? L’expérience conduite ici permet-elle d’affirmer que le `weight_decay` améliore toujours un MLP — formulez une conclusion exactement proportionnée aux observations dont vous disposez. Pourquoi enfin l’intervalle d’exactitude affiché ne suffit-il pas à établir la supériorité générale d’une architecture ?

Rédigez la deuxième réponse avec un soin particulier : elle constitue le modèle de phrase que vous aurez à écrire chaque fois que vous rapporterez un résultat expérimental, dans ce module comme ailleurs.


<a id="tp-e"></a>
## TP E — Quand l’information prend la forme d’une loi physique · 30 min

Jusqu’ici, chaque contrainte d’apprentissage provenait d’un couple entrée–étiquette. Nous conservons maintenant le même mécanisme différentiable, mais nous changeons la nature de l’information : une équation différentielle et des conditions initiales vont définir ce qu’est une prédiction acceptable. Aucune valeur de la solution ne sera fournie au réseau.

### Le problème, et pourquoi il a été choisi

Nous cherchons la solution de

$$
\ddot x(t)+2\gamma\dot x(t)+\omega_0^2x(t)=0,
\qquad x(0)=1,\quad \dot x(0)=0,
$$

avec $\gamma=0.3\;\mathrm{s}^{-1}$, $\omega_0=2\;\mathrm{s}^{-1}$ et $0\le t\le T=3\;\mathrm{s}$. Dans le régime sous-amorti, en posant $\omega_d=\sqrt{\omega_0^2-\gamma^2}$, la solution s’écrit

$$
x_\star(t)=e^{-\gamma t}\left[x_0\cos(\omega_dt)
+\frac{v_0+\gamma x_0}{\omega_d}\sin(\omega_dt)\right].
$$

Ce problème a été retenu parce qu’il est assez simple pour être vérifié de façon indépendante, par la formule analytique et par un solveur classique. La solution exacte et `solve_ivp` seront donc utilisés **uniquement pour le contrôle final**, jamais comme cibles d’apprentissage.

### Les deux ingrédients à comprendre avant de coder

Le réseau reçoit la variable sans dimension $s=t/T\in[0,1]$. La règle de chaîne impose alors $\dot x=u_s/T$ et $\ddot x=u_{ss}/T^2$ : normaliser l’entrée sans transformer les dérivées reviendrait à résoudre un autre problème physique, avec d’autres constantes de temps. Retenez ces deux facteurs, ils reviendront dans l’exercice E1.

Pour satisfaire exactement les conditions initiales, plutôt que de les pénaliser, nous paramétrons la sortie sous la forme

$$u_\theta(s)=x_0+Tv_0s+s^2N_\theta(s),$$

où $N_\theta$ est le réseau proprement dit. L’enveloppe $s^2$ garantit que la valeur et la dérivée en $s=0$ ne dépendent pas des paramètres : les conditions initiales sont vraies par construction, à la précision machine près, et non approchées par un compromis d’optimisation. La perte repose alors sur le seul résidu normalisé

$$
\widetilde r_\theta(s)=\frac{u_{ss}}{\omega_0^2T^2}
+\frac{2\gamma u_s}{\omega_0^2T}+u_\theta(s).
$$

Cette écriture est celle de l’équation divisée par $\omega_0^2$, ce qui donne au résidu l’unité d’un déplacement et rend la perte moins dépendante de l’échelle des constantes. Les coefficients physiques sont connus et ne sont pas optimisés.


In [ ]:
GAMMA, OMEGA0, T_FINAL = 0.3, 2.0, 3.0
X0, V0 = 1.0, 0.0
PINN_DTYPE = torch.float64

def exact_solution(t):
    omega_d = math.sqrt(OMEGA0**2 - GAMMA**2)
    return torch.exp(-GAMMA * t) * (X0 * torch.cos(omega_d * t)
           + (V0 + GAMMA * X0) / omega_d * torch.sin(omega_d * t))

class OscillatorPINN(nn.Module):
    def __init__(self, width=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1, width), nn.Tanh(),
                                 nn.Linear(width, width), nn.Tanh(), nn.Linear(width, 1))
    def forward(self, s):
        return X0 + T_FINAL * V0 * s + s.square() * self.net(s)

torch.manual_seed(SEED + 50)
pinn = OscillatorPINN().to(dtype=PINN_DTYPE, device=DEVICE)
s_collocation = torch.linspace(0., 1., 80, dtype=PINN_DTYPE).reshape(-1, 1)
print("Paramètres du PINN :", sum(p.numel() for p in pinn.parameters()))
print("Nombre de points de collocation :", len(s_collocation))


### E1 — Différencier le réseau par rapport à son entrée · 10 min

**Objectif.** Obtenir les dérivées première et seconde de la sortie du réseau par rapport à son entrée, et construire le résidu physique sans perdre les facteurs de la règle de chaîne.

**Ce que vous devez écrire.** Dans la fonction `derivatives_and_residual`, l’entrée `s` est déjà préparée avec `requires_grad_(True)` et la sortie `u` est déjà calculée. Il vous reste à obtenir `du`, c’est-à-dire $u_s$, puis `d2u`, c’est-à-dire $u_{ss}$, et enfin `residual`, égal à $\widetilde r_\theta(s)$ ; les quatre objets `u, du, d2u, residual` sont ensuite renvoyés, tous de forme $(M,1)$ comme `s`.

**Marche à suivre.** La fonction `torch.autograd.grad(sortie, entrée, ...)` renvoie un **tuple**, dont il faut prendre le premier élément avec `[0]`. Deux arguments sont indispensables ici. Le premier, `grad_outputs=torch.ones_like(u)`, indique que l’on dérive la somme des sorties : comme le réseau traite chaque ligne indépendamment, la dérivée de cette somme coïncide exactement avec les dérivées point par point recherchées — commodité qui ne se transposerait pas à un modèle mélangeant les observations d’un lot. Le second, `create_graph=True`, demande que l’opération de dérivation soit elle-même enregistrée dans le graphe ; sans lui, le second appel serait impossible et, surtout, la perte construite à partir des dérivées ne pourrait plus être différentiée par rapport aux paramètres. Le second appel est de même forme que le premier, en remplaçant `u` par `du`. Écrivez enfin le résidu en traduisant terme à terme la formule encadrée plus haut, en prenant garde aux dénominateurs $\omega_0^2T^2$ et $\omega_0^2T$.

**Vérification.** La cellule suivante applique votre fonction à un modèle de référence qui renvoie la solution analytique : son résidu normalisé doit être nul à $10^{-10}$ près, ce qui valide simultanément vos facteurs de chaîne et vos dénominateurs. Si ce contrôle échoue, l’erreur est dans le résidu et non dans le réseau. Elle vérifie ensuite qu’avant tout entraînement le réseau satisfait déjà les conditions initiales, conséquence de l’enveloppe $s^2$.

**Erreurs fréquentes.** Oublier `[0]` et manipuler un tuple ; omettre `create_graph=True`, ce qui produit une erreur au second appel ou, plus insidieusement, un entraînement qui ne progresse pas ; utiliser `u.backward()` au lieu de `torch.autograd.grad`, ce qui remplirait les `.grad` des paramètres sans fournir la dérivée par rapport à l’entrée ; oublier un facteur $T$ et obtenir un résidu de référence non nul, de l’ordre de l’unité.

**Un mot sur l’activation.** `Tanh` possède les dérivées régulières que cette formulation exige. Un MLP à activation ReLU est affine par morceaux : sa dérivée seconde classique est nulle presque partout et indéfinie aux ruptures, ce qui le rend inadapté à une formulation forte d’une équation différentielle du second ordre.


In [ ]:
def derivatives_and_residual(model, s_values):
    s = s_values.detach().clone().requires_grad_(True)
    u = model(s)
    # TODO E1 — deux dérivations automatiques, puis le résidu physique.
    #
    # Gabarit d'un appel :
    #   torch.autograd.grad(<sortie>, <entrée>, grad_outputs=torch.ones_like(<sortie>),
    #                       create_graph=True)[0]
    #   du  : dériver u  par rapport à s   -> forme (M, 1)
    #   d2u : dériver du par rapport à s   -> forme (M, 1)
    #
    # residual : traduire la formule de l'énoncé, équation divisée par OMEGA0**2.
    #   trois termes, dénominateurs OMEGA0**2 * T_FINAL**2 puis OMEGA0**2 * T_FINAL,
    #   et le terme en u sans dénominateur.
    raise NotImplementedError("E1 : deux dérivées automatiques et règle de la chaîne")
    return u, du, d2u, residual


In [ ]:
class ExactReference(nn.Module):
    def forward(self, s):
        return exact_solution(T_FINAL * s)

_, _, _, exact_residual = derivatives_and_residual(ExactReference(), s_collocation)
assert float(exact_residual.detach().abs().max()) < 1e-10
u0, du0, _, _ = derivatives_and_residual(pinn, torch.zeros(1, 1, dtype=PINN_DTYPE))
assert abs(float(u0.detach()) - X0) < 1e-12
assert abs(float(du0.detach()) / T_FINAL - V0) < 1e-12
print("Résidu normalisé de la solution exacte :", float(exact_residual.detach().abs().max()))
print("Conditions initiales imposées avant entraînement :", float(u0.detach()), float(du0.detach()) / T_FINAL)


### E2 — Optimiser le résidu sans confondre méthode et garantie · 12 min

La procédure d’optimisation est fournie ; lisez-la avant de l’exécuter, car elle illustre une pratique courante dans la littérature sur les PINN. Adam effectue une première phase de descente sur 1200 pas, puis L-BFGS affine la solution sur le même ensemble fixe de points de collocation. Cette combinaison est efficace sur ce petit problème, mais elle ne constitue pas une recette universelle. Notez que L-BFGS peut appeler plusieurs fois sa fermeture au cours d’une même itération : le nombre d’itérations et le nombre d’évaluations de la perte ne coïncident donc pas, ce que la sortie affichée rend visible.

Le critère ne contient que le résidu différentiel, puisque la paramétrisation impose déjà les conditions initiales. Cette remarque mérite qu’on s’y arrête : sans condition initiale, la fonction identiquement nulle satisferait elle aussi l’équation. Une loi différentielle seule ne sélectionne pas l’expérience physique que l’on veut représenter ; c’est la conjonction de l’équation et des conditions au bord qui détermine une trajectoire. Pendant l’exécution, qui demande quelques dizaines de secondes, observez la décroissance de la perte affichée et demandez-vous ce qu’elle mesure exactement — et surtout, ce qu’elle ne mesure pas.


In [ ]:
def fit_pinn(model, points, adam_steps=1200, lbfgs_iterations=200):
    trace = []
    adam = torch.optim.Adam(model.parameters(), lr=0.002)
    model.train()
    for step in range(adam_steps):
        adam.zero_grad(set_to_none=True)
        _, _, _, residual = derivatives_and_residual(model, points)
        loss = residual.square().mean()
        loss.backward()
        adam.step()
        trace.append(float(loss.detach()))
    lbfgs = torch.optim.LBFGS(model.parameters(), lr=1.0,
                             max_iter=lbfgs_iterations, max_eval=350,
                             tolerance_grad=1e-10, tolerance_change=1e-12,
                             line_search_fn="strong_wolfe")
    closure_values = []
    def closure():
        lbfgs.zero_grad(set_to_none=True)
        _, _, _, residual = derivatives_and_residual(model, points)
        loss = residual.square().mean()
        loss.backward()
        closure_values.append(float(loss.detach()))
        return loss
    lbfgs.step(closure)
    model.eval()
    return trace, closure_values

pinn_start = time.perf_counter()
pinn_trace, lbfgs_trace = fit_pinn(pinn, s_collocation)
print(f"Entraînement PINN : {time.perf_counter() - pinn_start:.1f} s sur cet environnement")
print("Évaluations Adam / fermeture L-BFGS :", len(pinn_trace), len(lbfgs_trace))
print("Dernière perte Adam / dernière perte L-BFGS :", pinn_trace[-1], lbfgs_trace[-1])


### E3 — Contrôler la solution ailleurs que là où elle a été ajustée · 8 min

Nous évaluons maintenant le réseau sur **400 points intercalés, distincts des points de collocation**, sans réentraîner après inspection — l’assertion placée en tête de la cellule vérifie d’ailleurs que ces points sont bien disjoints de ceux de l’optimisation. Cette précaution est l’exact analogue, pour un PINN, du jeu de test scellé du TP D : une perte de collocation faible garantit seulement que l’équation est satisfaite là où on l’a demandé.

**Grille de lecture.** Trois contrôles répondent à trois questions différentes, et il faut se garder de les confondre. Le premier demande si la trajectoire est proche d’une référence indépendante, ce que mesurent l’erreur $L^2$ relative et l’erreur maximale. Le deuxième demande si l’équation est satisfaite **entre** les points utilisés par l’optimisation, ce que mesure la moyenne quadratique du résidu physique. Le troisième demande si les conditions initiales sont effectivement respectées, ce que mesurent les deux dernières entrées du rapport. Retenez que la racine de la moyenne des carrés du résidu physique s’exprime en déplacement par seconde carrée, tandis que le résidu normalisé possède l’unité d’un déplacement ; et qu’une petite moyenne quadratique sur un ensemble fini de points ne constitue jamais, à elle seule, une borne uniforme de l’erreur de solution.

**Un point technique à ne pas manquer.** Pour calculer uniquement des valeurs de la fonction, `torch.no_grad()` conviendrait. Pour contrôler un résidu différentiel, le suivi des gradients par rapport à l’entrée doit rester actif, même après un appel à `model.eval()`. C’est ici que se paie concrètement la distinction établie au TP C : le mode d’évaluation des couches et l’activation de l’autodifférentiation sont deux dimensions indépendantes du calcul. La cellule suivante ne contient donc aucun `no_grad`, et ce n’est pas un oubli.

La dernière cellule de cette partie compare enfin le résultat à un solveur classique de haute précision. Lisez le temps de calcul affiché avec prudence : il illustre le cas présent et ne constitue pas un banc d’essai général.


In [ ]:
s_check = ((torch.arange(400, dtype=PINN_DTYPE) + .5) / 400).reshape(-1, 1)
assert torch.cdist(s_check, s_collocation).min() > 1e-8
# Pas de no_grad ici : les dérivées par rapport à l'entrée sont nécessaires.
u_check, _, _, normalized_residual = derivatives_and_residual(pinn, s_check)
t_check = T_FINAL * s_check
reference = exact_solution(t_check)
relative_l2 = float((torch.linalg.vector_norm(u_check - reference)
                     / torch.linalg.vector_norm(reference)).detach())
max_error = float((u_check - reference).detach().abs().max())
residual_rms = float((OMEGA0**2 * normalized_residual.detach()).square().mean().sqrt())
u0, du0, _, _ = derivatives_and_residual(pinn, torch.zeros(1, 1, dtype=PINN_DTYPE))
pinn_report = {"relative_l2": relative_l2, "max_error": max_error,
               "physical_residual_rms": residual_rms,
               "initial_position_error": abs(float(u0.detach()) - X0),
               "initial_velocity_error": abs(float(du0.detach()) / T_FINAL - V0)}
print(pinn_report)
assert all(np.isfinite(v) for v in pinn_report.values())

fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
axes[0].plot(t_check.numpy(), reference.numpy(), color="black", label="Solution analytique")
axes[0].plot(t_check.numpy(), u_check.detach().numpy(), "--", label="PINN")
axes[0].set(xlabel="Temps (s)", ylabel="Déplacement", title="Solution hors collocation")
axes[0].legend(fontsize=8)
axes[1].plot(t_check.numpy(), (OMEGA0**2 * normalized_residual).detach().numpy())
axes[1].set(xlabel="Temps (s)", ylabel="Résidu physique", title="Équation contrôlée ailleurs")
axes[2].semilogy(np.arange(1, len(pinn_trace)+1), pinn_trace, label="Adam")
axes[2].semilogy(np.arange(len(pinn_trace)+1, len(pinn_trace)+len(lbfgs_trace)+1), lbfgs_trace, label="L-BFGS")
axes[2].set(xlabel="Évaluation de perte", ylabel="Moyenne du résidu normalisé²", title="Optimisation")
axes[2].legend(fontsize=8)
fig.tight_layout()
export_figure(fig, "pinn_controls")
plt.show()


In [ ]:
# Référence numérique classique : contrôle du PINN et de la formule analytique.
from scipy.integrate import solve_ivp
solver_start = time.perf_counter()
t_numpy = t_check[:, 0].numpy()
sol_ivp = solve_ivp(lambda t, z: [z[1], -2*GAMMA*z[1] - OMEGA0**2*z[0]],
                    (0., T_FINAL), [X0, V0], t_eval=t_numpy,
                    method="DOP853", rtol=1e-10, atol=1e-12)
assert sol_ivp.success
ivp_seconds = time.perf_counter() - solver_start
ivp_error = np.max(np.abs(sol_ivp.y[0] - reference[:, 0].numpy()))
print(f"solve_ivp : {ivp_seconds:.4f} s ; écart maximal à la formule analytique : {ivp_error:.3e}")
print("Ces chronométrages illustrent le cas présent ; ils ne constituent pas un benchmark général.")


### Bilan du TP E — préciser ce qui a réellement été appris

Pourquoi la sortie identiquement nulle est-elle exclue par notre paramétrisation, et que se passerait-il si l’on avait choisi $x_0=0$ et $v_0=0$ ? Quel facteur de conversion serait perdu si l’on confondait une dérivée par rapport à $s$ avec une dérivée par rapport à $t$ ? Pourquoi `create_graph=True` est-il nécessaire pendant l’entraînement, alors qu’il ne le serait pas pour un simple diagnostic ? Que démontre enfin cette expérience contrôlée sur la construction d’un PINN, et que ne démontre-t-elle pas sur sa supériorité supposée face aux méthodes numériques classiques ?


<a id="sortie"></a>
## Ticket de sortie · 5 min

Répondez sans relancer le moindre calcul, puis confrontez vos réponses avec le binôme voisin. Le but est de reformuler la séance comme une chaîne de décisions, et non comme une succession de commandes PyTorch.

La première situation est la suivante : la perte d’entraînement est faible, mais la perte de validation demeure forte ; citez deux hypothèses de nature différente à examiner, et le contrôle qui permettrait de les départager. Ensuite, une sortie de forme `(64, 2)` et des cibles de forme `(64,)` sont-elles compatibles avec l’entropie croisée, et quel doit être le type de ces cibles ? Un appel à `backward()` modifie-t-il directement les poids ? Pourquoi un hyperparamètre choisi grâce au test interdit-il de présenter ensuite ce même test comme une évaluation indépendante ? Dans le PINN, quels rôles distincts jouent l’équation différentielle et les conditions initiales ? Que vous a enfin appris le quartet d’Anscombe que ne pouvait pas montrer le seul tableau de statistiques ?

**Trace à conserver.** Une figure commentée du quartet d’Anscombe, une frontière de décision, le rapport de sélection et de test, les trois contrôles du PINN, et six phrases répondant aux questions ci-dessus. Chaque résultat doit pouvoir être expliqué sans invoquer le réseau ou le logiciel comme une autorité.


## Prolongements facultatifs — pour transformer le TP en véritable étude

Ces activités dépassent les cinq heures. Elles sont proposées pour prolonger la méthode expérimentale, non pour accumuler des architectures.

**1. Mesurer la variabilité d’une comparaison.** Répéter le protocole complet sur cinq graines avec un budget fixé, puis résumer moyenne et dispersion des pertes de validation. Distinguer la variabilité du réentraînement de l’incertitude sur un test fini. Ne pas recycler les 180 observations déjà ouvertes pour poursuivre les réglages.

**2. Inspecter le comportement des couches.** Introduire un dropout entre les couches cachées et comparer des prédictions répétées en modes `train` et `eval`, avec puis sans enregistrement des gradients. Prévoir les quatre cas avant de les exécuter. La dispersion induite par dropout ne constitue pas automatiquement une incertitude probabiliste calibrée.

**3. Dégrader le PINN de manière contrôlée.** Allonger l’horizon, raréfier les points de collocation ou remplacer `Tanh` par ReLU. Repartir d’une nouvelle instance et conserver une expérience de référence. Comparer conjointement l’erreur de trajectoire et le résidu sur une grille indépendante. Avec l’enveloppe $s^2$, la sortie complète peut conserver de la courbure, mais l’autodifférentiation point par point ne représente pas les contributions singulières aux ruptures d’un réseau ReLU : la formulation forte demeure délicate.

**4. Passer à un problème inverse.** Rendre $\gamma$ et $\omega_0$ inconnus et ajouter des mesures bruitées de $x(t)$. Construire une perte combinant données et dynamique, puis étudier identifiabilité, sensibilité et robustesse à l’initialisation. Une contrainte de positivité peut être imposée par `softplus`, mais elle ne garantit pas que les observations contiennent assez d’information pour identifier les paramètres.

**5. Revenir au quartet d’Anscombe.** Réentraîner une régression après retrait successif d’un point. Comparer en particulier le troisième et le quatrième jeu : un point atypique en $y$ et un point de fort levier en $x$ n’influencent pas la droite de la même manière. Relier cette expérience aux notions de robustesse, de diagnostic des résidus et de décalage de distribution.


<a id="optimisateurs"></a>
## Complément expérimental — SGD, RMSprop, Adam et AdamW · hors séance

Ce complément prolonge le chapitre du cours consacré aux optimiseurs ; il ne constitue pas un sixième TP obligatoire. La même discipline que pour les données s’applique : annoncer la grandeur que l’on compare, modifier un seul mécanisme à la fois et résister à la tentation de désigner un vainqueur à partir d’une trajectoire isolée.

### O1 — Pourquoi corriger le démarrage d’Adam ?

Avec $m_0=v_0=0$, le premier gradient donne $m_1=(1-\beta_1)g_1$ et $v_1=(1-\beta_2)g_1^2$. Les estimateurs corrigés sont $\widehat m_1=g_1$ et $\widehat v_1=g_1^2$. L’expression « correction du biais » désigne ici la masse manquante des moyennes exponentielles initialisées à zéro ; elle ne transforme pas la direction complète d’Adam en estimateur sans biais d’un gradient stationnaire idéal.

**Question.** Pour $g_1=2$, $\beta_1=0.9$, $\beta_2=0.999$ et $\alpha=0.1$, calculez la mise à jour avec et sans correction, d’abord en négligeant $\varepsilon$, puis vérifiez numériquement. RMSprop usuel conserve lui aussi une moyenne des carrés, mais sans cette correction de démarrage.


In [ ]:
g1 = torch.tensor(2., dtype=torch.float64)
beta1, beta2, alpha, eps = .9, .999, .1, 1e-8
m1, v1 = (1-beta1)*g1, (1-beta2)*g1.square()
m1_hat, v1_hat = m1/(1-beta1), v1/(1-beta2)
uncorrected_step = alpha * m1 / (v1.sqrt() + eps)
corrected_step = alpha * m1_hat / (v1_hat.sqrt() + eps)
print(f"m1={m1:.4f}, v1={v1:.4f}, m1 corrigé={m1_hat:.4f}, v1 corrigé={v1_hat:.4f}")
print(f"Quantité soustraite : sans correction {uncorrected_step:.6f} ; Adam corrigé {corrected_step:.6f}")


### O2 — Une vallée étroite éclaire la géométrie, pas la généralisation

Nous minimisons la quadratique $f(\theta)=\tfrac12\theta^\top H\theta$, dont les valeurs propres sont 1 et 50. Une rotation de 30° rend les directions propres obliques aux axes des paramètres. Tous les algorithmes partent de $(3,3)$ et disposent de 250 mises à jour, mais leurs taux sont explicitement adaptés à la nature de leur règle : imposer la même valeur numérique à des mises à jour différentes ne constitue pas automatiquement une comparaison équitable.

SGD utilise un pas 0.03, inférieur à $2/\lambda_{\max}=0.04$ ; RMSprop utilise 0.07 avec `alpha=0.9`, sans momentum ; Adam et AdamW utilisent 0.08 avec $(\beta_1,\beta_2)=(0.9,0.999)$. AdamW ajoute un `weight_decay` de 0.03 et n’effectue donc pas exactement la même optimisation. La grandeur représentée reste toutefois la **même perte de données**.

**Questions.** Que limite un préconditionnement diagonal lorsque les directions principales sont tournées ? Une descente rapide sur cette quadratique prédit-elle la généralisation d’un réseau ? Pourquoi un calendrier de taux pourrait-il modifier l’allure des trajectoires ?


In [ ]:
angle = math.pi / 6
rotation = torch.tensor([[math.cos(angle), -math.sin(angle)],
                          [math.sin(angle), math.cos(angle)]], dtype=torch.float64)
H = rotation @ torch.diag(torch.tensor([1., 50.], dtype=torch.float64)) @ rotation.T

def quadratic(theta):
    return .5 * theta @ H @ theta

optimizer_factories = {
    "SGD, lr=.03": lambda params: torch.optim.SGD(params, lr=.03),
    "RMSprop, lr=.07": lambda params: torch.optim.RMSprop(params, lr=.07, alpha=.9, eps=1e-8, momentum=0.),
    "Adam, lr=.08": lambda params: torch.optim.Adam(params, lr=.08, betas=(.9, .999), eps=1e-8),
    "AdamW, lr=.08, wd=.03": lambda params: torch.optim.AdamW(params, lr=.08, betas=(.9, .999), eps=1e-8, weight_decay=.03),
}
optimizer_results = {}
for name, factory in optimizer_factories.items():
    theta = nn.Parameter(torch.tensor([3., 3.], dtype=torch.float64))
    optimizer = factory([theta])
    trajectory = [theta.detach().clone()]
    values = [float(quadratic(theta).detach())]
    for step in range(250):
        optimizer.zero_grad(set_to_none=True)
        value = quadratic(theta)
        value.backward()
        optimizer.step()
        trajectory.append(theta.detach().clone())
        values.append(float(quadratic(theta).detach()))
    optimizer_results[name] = {"trajectory": torch.stack(trajectory).numpy(), "loss": np.array(values)}
    print(f"{name:28s} | perte finale {values[-1]:.6g}")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0))
grid = np.linspace(-1.5, 4., 220)
qx, qy = np.meshgrid(grid, grid)
coords = np.stack([qx, qy], axis=-1)
qvalues = .5 * np.einsum("...i,ij,...j->...", coords, H.numpy(), coords)
axes[0].contour(qx, qy, qvalues, levels=[.1, 1, 5, 20, 80, 200], colors="0.8", linewidths=.8)
for name, result in optimizer_results.items():
    trajectory = result["trajectory"]
    axes[0].plot(trajectory[:, 0], trajectory[:, 1], label=name, linewidth=1.3)
    axes[1].semilogy(result["loss"], label=name)
axes[0].scatter([0], [0], marker="*", color="black", s=80)
axes[0].set(xlabel=r"$\theta_1$", ylabel=r"$\theta_2$", title="Trajectoires : vallée quadratique", aspect="equal")
axes[1].set(xlabel="Pas d'optimisation", ylabel="Perte de données", title="Même point initial, 250 pas")
axes[1].legend(fontsize=7)
axes[1].grid(alpha=.2)
fig.tight_layout()
export_figure(fig, "optimizer_trajectories")
plt.show()


### O3 — Pénalisation quadratique dans Adam et décroissance AdamW

Une pénalité $\lambda\lVert\theta\rVert^2/2$ ajoute $\lambda\theta$ au gradient **avant** la mise à jour des moments d’Adam. AdamW calcule les moments à partir du gradient de données, puis applique séparément la contraction $\theta\leftarrow(1-\alpha\lambda)\theta$. La décroissance découplée et la correction du démarrage répondent donc à deux questions entièrement différentes.

Le premier pas ci-dessous part de $\theta=(1,2)$, avec $g=(-0.05,0.5)$, $\alpha=0.1$ et $\lambda=0.1$. Dans la version couplée, le premier gradient change même de signe. Nous injectons ces gradients connus afin d’isoler les règles de mise à jour ; il ne s’agit pas d’un nouvel entraînement sur le jeu XOR.


In [ ]:
from IPython.display import display, Markdown
initial = torch.tensor([1., 2.], dtype=torch.float64)
data_gradient = torch.tensor([-.05, .5], dtype=torch.float64)
regularization = .1
rows = []
for method in ["Adam sans régularisation", "Adam + pénalité L2", "AdamW"]:
    theta = nn.Parameter(initial.clone())
    if method == "AdamW":
        opt = torch.optim.AdamW([theta], lr=.1, betas=(.9, .999), eps=1e-8, weight_decay=regularization)
        theta.grad = data_gradient.clone()
    else:
        opt = torch.optim.Adam([theta], lr=.1, betas=(.9, .999), eps=1e-8, weight_decay=0.)
        theta.grad = data_gradient.clone()
        if method == "Adam + pénalité L2":
            theta.grad += regularization * theta.detach()
    opt.step()
    rows.append((method, theta.detach().tolist()))
lines = ["| Méthode | θ₁ après un pas | θ₂ après un pas |", "|---|---:|---:|"]
lines.extend(f"| {name} | {value[0]:.6f} | {value[1]:.6f} |" for name, value in rows)
display(Markdown("\n".join(lines)))


### Interprétation du complément

Expliquez, avec vos propres mots, la différence entre :

- la mémoire du gradient ;
- la mémoire du carré du gradient ;
- la correction du démarrage des moyennes exponentielles ;
- la décroissance des poids.

Citez ensuite au moins quatre éléments à consigner pour rendre une comparaison reproductible : taux d’apprentissage, coefficients de mémoire, $\varepsilon$, décroissance, taille des lots, ordre des données, initialisation, budget et calendrier de taux sont des exemples possibles.


## Sources et documentation pour poursuivre

Le polycopié joint donne la bibliographie scientifique complète. Les références suivantes sont directement liées aux opérations réalisées dans ce notebook.

- F. J. Anscombe, [*Graphs in Statistical Analysis*](https://doi.org/10.1080/00031305.1973.10478966), *The American Statistician* 27(1), 1973 : quatre jeux de données aux statistiques proches, conçus pour montrer la nécessité de représenter les observations.
- PyTorch, [Tensor views](https://docs.pytorch.org/docs/stable/tensor_view.html) et [broadcasting semantics](https://docs.pytorch.org/docs/stable/notes/broadcasting.html) : formes, indexation et diffusion des dimensions.
- PyTorch, [CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) : logits et format des cibles.
- PyTorch, [Autograd mechanics](https://docs.pytorch.org/docs/stable/notes/autograd.html) : graphe de calcul, accumulation des gradients et mécanismes d’évaluation.
- PyTorch, [gradcheck](https://docs.pytorch.org/docs/stable/generated/torch.autograd.gradcheck.html) : contrôle numérique des dérivées en double précision.
- D. P. Kingma et J. Ba, [*Adam: A Method for Stochastic Optimization*](https://arxiv.org/abs/1412.6980), ICLR 2015.
- I. Loshchilov et F. Hutter, [*Decoupled Weight Decay Regularization*](https://openreview.net/forum?id=Bkg6RiCqY7), ICLR 2019.
- M. Raissi, P. Perdikaris et G. E. Karniadakis, [*Physics-informed neural networks*](https://doi.org/10.1016/j.jcp.2018.10.045), *Journal of Computational Physics* 378, 2019.
- A. S. Krishnapriyan et al., [*Characterizing possible failure modes in physics-informed neural networks*](https://proceedings.neurips.cc/paper/2021/hash/df438e5206f31600e6ae4af72f2725f1-Abstract.html), NeurIPS 2021.
- S. Wang et al., [*An Expert’s Guide to Training Physics-informed Neural Networks*](https://arxiv.org/abs/2308.08468), 2023.

Les API PyTorch sont documentées en ligne et peuvent évoluer. Pour une expérience reproductible, consignez les versions effectivement utilisées, les graines, le matériel, le budget et les options des optimiseurs.


In [ ]:
print(f"Durée totale des calculs, hors interactions : {time.perf_counter() - NOTEBOOK_START:.1f} s")
